In [ ]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml import paths
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 


from ast import For
from cmath import exp

from titanic_ml.common.experiments.runner import run_experiments, run_experiment_group_workflow
from titanic_ml.common.experiments.config import ALL_EXPERIMENTS
from titanic_ml.common.experiments.report import experiment_report, experiment_group_summary_report, baseline_summary_to_markdown, workflow_report
from titanic_ml.common.experiments.save_load import save_results, load_results, save_configs, load_configs
from titanic_ml.common.experiments.compare import leaderboard, compare_experiment_groups, summarize_group_comparison, titanic_notes_leaderboard, model_progression
from titanic_ml.common.models.registry import MODEL_REGISTRY
from titanic_ml.feature_engineering import add_family_features, add_has_cabin, add_title, add_full_title_feature



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
TARGET = "Survived"
ALL_EXPERIMENTS = ALL_EXPERIMENTS
for experiment_name, exp_config in ALL_EXPERIMENTS.items():
    print(f"Experiment: {experiment_name}")

train_df = pd.read_csv(paths.TRAIN_PATH)

exp_configs = ALL_EXPERIMENTS["fe05__title"]

# Line to rerun all experiments to update the results with the latest code changes. This will take a while.
# Uncomment to run all experiments and update results.
# for Name, exp_config in ALL_EXPERIMENTS.items():
#     print(f"Running {Name} experiments...")
#     exp_result = run_experiments(train_df, exp_config, target=TARGET, verbose=True, debug=True)
#     save_results(exp_result)
#     save_configs(exp_config)



Experiment: baseline__raw
Experiment: fe01__family
Experiment: fe02__has_cabin
Experiment: fe03__deck
Experiment: fe04__cabin_features
Experiment: fe05__title


In [3]:
# print("Experiment Configurations:")
# print(exp_configs)
# for exp_config in exp_configs:
#     print(exp_config)

In [4]:
# Work flow for running an experiment group, comparing it to the baseline, and generating a report. 
# This is the main workflow for analyzing the results of an experiment group and generating insights from it.
workflow = run_experiment_group_workflow(
    df=train_df,
    experiment_configs=exp_configs,
    target=TARGET,
)

print("Workflow completed. Here are the results:")
print("Comparison between baseline and feature engineering group:")
print(workflow["comparison"])
print("Summary of comparison:")
print(workflow["summary"])
print("Leaderboard:")
print(workflow["leaderboard"])

running exp: {'name': 'fe05__title__logreg', 'features': ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title'], 'feature_engineering': [<function add_full_title_feature at 0x000001E6FCEFC4C0>], 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare'], 'onehot_features': ['Sex', 'Embarked', 'Title'], 'ordinal_features': ['Pclass'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': "Feature engineering 03: adds Title feature, which extracts the title from the passenger's name. This is expected to give more information about the passenger's social status, which can be a strong signal for survival. The title is extracted and then grouped into common titles and a 'Rare' category fo

In [5]:
# Generate a full report for the workflow, including the comparison, summary, and leaderboard. 
# This will be a markdown report that can be easily shared and visualized.
# Mainly used for generating the report for the notebook, but can also be used for generating reports for individual experiment groups or comparisons.
full_report = workflow_report(workflow)

print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])

Full workflow report:

Report
### fe05__title

<details>
<summary>Experiment details</summary>

_Description pending._

<details>
<summary>Comparison details</summary>

#### Comparison vs baseline__raw

| reference_group   | compare_group   | model_name    |   test_accuracy_mean_reference |   test_accuracy_mean_compare |   test_accuracy_mean_delta |   test_f1_mean_reference |   test_f1_mean_compare |   test_f1_mean_delta |
|:------------------|:----------------|:--------------|-------------------------------:|-----------------------------:|---------------------------:|-------------------------:|-----------------------:|---------------------:|
| baseline__raw     | fe05__title     | logreg        |                          0.786 |                        0.825 |                      0.039 |                    0.713 |                  0.765 |                0.052 |
| baseline__raw     | fe05__title     | knn           |                          0.809 |                        0.822 |      

In [6]:
# Reminder of how to run a single experiment if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# exp_config = [exp_config]
# exp_result = run_experiments(train_df, exp_config, target=TARGET,)
# exp_report = experiment_report(exp_result, exp_config, print_report=True)
# save_results(exp_result)
# save_configs(exp_config)

In [7]:
# Summary for baseline experiment to use in report, 
# since we can't compare it to itself.

# baseline_summary = baseline_summary_to_markdown(exp_result)
# print("Baseline summary:")
# print(baseline_summary)

In [8]:
# Reminder for how to load results and configs if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# all_results = load_results()
# print("Loaded results:")
# print(all_results)

# all_configs = load_configs()
# print("Loaded configs:")
# print(all_configs)

In [19]:
print(workflow["all_results"])

                             experiment     stage   feature_group  \
0                 baseline__raw__logreg  baseline             raw   
1                    baseline__raw__knn  baseline             raw   
2                    baseline__raw__svc  baseline             raw   
3          baseline__raw__decision_tree  baseline             raw   
4          baseline__raw__random_forest  baseline             raw   
5            baseline__raw__extra_trees  baseline             raw   
6                    baseline__raw__xgb  baseline             raw   
7                  fe01__family__logreg      fe01          family   
8                     fe01__family__knn      fe01          family   
9                     fe01__family__svc      fe01          family   
10          fe01__family__decision_tree      fe01          family   
11          fe01__family__random_forest      fe01          family   
12            fe01__family__extra_trees      fe01          family   
13                    fe01__family

In [21]:
for model in MODEL_REGISTRY:
    model_progression_df = model_progression(workflow["all_results"], model_name=model, metric="test_accuracy_mean")
    print(f"Model progression for {model}:")
    print(model_progression_df)
    print()

Model progression for logreg:
      stage   feature_group  test_accuracy_mean
0  baseline             raw               0.786
1      fe01          family               0.795
2      fe02       has_cabin               0.791
3      fe03            deck               0.791
4      fe04  cabin_features               0.791
5      fe05           title               0.825

Model progression for knn:
      stage   feature_group  test_accuracy_mean
0  baseline             raw               0.809
1      fe01          family               0.805
2      fe02       has_cabin               0.809
3      fe03            deck               0.818
4      fe04  cabin_features               0.817
5      fe05           title               0.822

Model progression for svc:
      stage   feature_group  test_accuracy_mean
0  baseline             raw               0.827
1      fe01          family               0.826
2      fe02       has_cabin               0.825
3      fe03            deck               0.825
4 

In [25]:
print(train_df)

     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                                 ...     ...   ... 